# OPS Environment Tutorial

This notebook contains the tutorial for the 2025 IEEE BioCAS Grand Challenge - Closed-loop Neural Decoding for Motor Control of non-Human Primates.

For this Grand Challenge, we are performing a closed-loop simulation of a brain model (from hereon referred to as an OPS (Online Prosthesis Simulator)).

In this tutorial, we will go over how the OPS can be setup and used in a closed-loop environment, while using **NeuroBench** to evaluate the performance of the trained model.

## Import packages

For this notebook, you will need to install the NeuroBench version used for this Grand Challenge.
You can install the package with the following instruction:

```
pip install git+https://github.com/NeuroBench/neurobench.git@2025_GC
```

For other packages, please refer to the official NeuroBench for version compatibility.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import os
import sys

# sys.path.append("/home/vsun/closed_loop_test/")
from neurobench.envs import OPS, OPSEnv
from neurobench.models.torch_model import TorchModel
from neurobench.benchmarks import BenchmarkClosedLoop

from neurobench.metrics.workload import (
    ActivationSparsity,
    SynapticOperations,
)
from neurobench.metrics.static import (
    Footprint,
    ConnectionSparsity,
)

## Global Variable Definition

Here we define the where the model's weight and the OPS neurons is stored, along with some variable used for the closed-loop simulation

In [ ]:
model_path = "examples/closed_loop_ops/OPS_model_state_dict.pth"
neuron_file = "examples/closed_loop_ops/neuron1.csv"

# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")

time_step = 0.01
num_neurons=96
max_duration = 3.0
min_time_in_target = 0.5 # value in seconds
target_size = 2.5

## Declare OPS and OPS Environment

In [ ]:
ops = OPS(
    num_neurons=num_neurons,
    time_step=time_step,
    upper_lmax=100,
    lower_lmax=40,
    upper_lmin=5,
    zero_prob=0.5,
    device=device
)

In [ ]:
ops.assign_neurons(neuron_file)

In [ ]:
env = OPSEnv(
    ops=ops,
    max_duration=max_duration,
    min_time_in_target=min_time_in_target,
    side_radius=10,
    min_distance=8,
    target_size=target_size,
    device=device
)

In [ ]:
class ANNModel(nn.Module):
    def __init__(self, input_dim, layer1=16, layer2=32, output_dim=2,
                 bin_window=0.2, sampling_rate=0.004, drop_rate=0.5):
        super().__init__()

        self.input_dim = input_dim
        self.output_dim = output_dim
        self.layer1 = layer1
        self.layer2 = layer2
        self.drop_rate = drop_rate

        self.bin_window_time = bin_window
        self.sampling_rate = sampling_rate
        self.bin_window_size = int(self.bin_window_time / self.sampling_rate)

        self.fc1 = nn.Linear(self.input_dim, self.layer1)
        self.fc2 = nn.Linear(self.layer1, self.layer2)
        self.fc3 = nn.Linear(self.layer2, self.output_dim)
        self.dropout = nn.Dropout(self.drop_rate)
        self.batchnorm1 = nn.BatchNorm1d(self.layer1)
        self.batchnorm2 = nn.BatchNorm1d(self.layer2)
        self.activation = nn.ReLU()
        self.batch_size = 1

        self.register_buffer("data_buffer", torch.zeros(1, input_dim).type(torch.float32), persistent=False)

    def single_forward(self, x):
        x = self.activation(self.fc1(x.view(self.batch_size, -1)))
        x = self.activation(self.dropout(self.fc2(x)))
        x = self.fc3(x)

        return x.squeeze(dim=0)

    def forward(self, x):
        self.data_buffer = torch.cat((self.data_buffer, x), dim=0)
        self.data_buffer = self.data_buffer[1:, :]

        pred = self.single_forward(x)

        return pred

In [ ]:
net = ANNModel(input_dim=num_neurons)
net.load_state_dict(torch.load(model_path, map_location=device))
model = TorchModel(net)

In [ ]:
static_metrics = [Footprint, ConnectionSparsity]
workload_metrics = [ActivationSparsity, SynapticOperations]

In [ ]:
benchmark = BenchmarkClosedLoop(model, env, [], [], [static_metrics, workload_metrics])

In [ ]:
results = benchmark.run(nr_interactions=50, max_length=300, device=device)
print(results)